# Game Trial Design Checks

Encoding-trial mechanics: block conditions, drone/bag trajectories, change-points, fragment distribution, and reward scoring.

In [1]:
import re, json, hashlib, os, math, warnings, textwrap, sys
from pathlib import Path
from datetime import datetime
from collections import Counter, OrderedDict
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
from IPython.display import display, HTML

warnings.filterwarnings("ignore")
pd.set_option("display.max_rows", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

# Import shared JS parsers
sys.path.insert(0, str(Path(".").resolve()))
from js_import import (load_sources, parse, save_fig, save_html, apply_style,
                        REPO_ROOT, DESIGN_DIR, FIG_DIR, sha256)
apply_style()

# ── Load all JS sources ──
stimuli_src = (REPO_ROOT / "static/task/stimuli.js").read_text() if (REPO_ROOT / "static/task/stimuli.js").exists() else ""
memory_src  = (REPO_ROOT / "static/task/memory_task.js").read_text() if (REPO_ROOT / "static/task/memory_task.js").exists() else ""
details_src = (REPO_ROOT / "static/task/stimuli-details.js").read_text() if (REPO_ROOT / "static/task/stimuli-details.js").exists() else ""
trial_src   = (REPO_ROOT / "static/task/trial.js").read_text() if (REPO_ROOT / "static/task/trial.js").exists() else ""
index_src   = (REPO_ROOT / "index.html").read_text() if (REPO_ROOT / "index.html").exists() else ""

# Re-export original helper functions from the shared module
parse_js_flat_array = parse.flat_array
parse_js_2d_array = parse.array_2d
parse_js_string = parse.string
parse_js_string_array = parse.string_array
parse_js_int_pair_array = parse.int_pair_array

# ── Original helper functions that stay local ──
def read_text(relpath):
    p = REPO_ROOT / relpath
    return p.read_text(encoding="utf-8") if p.exists() else None

def pair_in(pair, plist):
    return any(pair[0] == p[0] and pair[1] == p[1] for p in plist)

def styled_table_css(uid="tbl"):
    return textwrap.dedent(f"""\
    <style>
    #{uid} th {{ background:#f3f3f3; color:#222; font-weight:650;
      border:1px solid #d6d6d6; padding:5px 10px; text-align:left; }}
    #{uid} td {{ border:1px solid #e1e1e1; padding:4px 10px;
      font-variant-numeric:tabular-nums; }}
    #{uid} tr:nth-child(even) td {{ background:#fafafa; }}
    #{uid} table {{ border-collapse:collapse; font-size:12px;
      font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Arial,sans-serif; }}
    #{uid} caption {{ caption-side:top; padding:4px 0; color:#444;
      font-size:12px; text-align:left; font-weight:600; }}
    </style>
    """)

# save_html is imported from js_import (category-aware)

print("Setup complete.")


Setup complete.


## C. Block condition design

In [2]:
factors_vol = parse_js_flat_array(stimuli_src, "factors_vol")
factors_stc = parse_js_flat_array(stimuli_src, "factors_stc")
fv_m = re.search(r"var\s+factors_valence\s*=\s*\[(.*?)\]", stimuli_src)
factors_valence = re.findall(r"'(\w+)'", fv_m.group(1)) if fv_m else []

block_df = pd.DataFrame({
    "block": [1,2,3,4],
    "valence": factors_valence,
    "vol_param": [int(v) for v in factors_vol],
    "vol_level": ["high" if v==49 else "low" for v in factors_vol],
    "stc_param": [int(s) for s in factors_stc],
    "stc_level": ["high" if s==64 else "low" for s in factors_stc],
})
display(block_df)

for blk, val, vol, stc in [(1,"reward",49,16),(2,"reward",4,64),(3,"loss",49,16),(4,"loss",4,64)]:
    r = block_df[block_df.block==blk].iloc[0]
    assert r.valence==val and r.vol_param==vol and r.stc_param==stc
print("\u2713 Block condition design matches specification.")

,block,valence,vol_param,vol_level,stc_param,stc_level
0,1,reward,49,high,16,low
1,2,reward,4,low,64,high
2,3,loss,49,high,16,low
3,4,loss,4,low,64,high


✓ Block condition design matches specification.


## D. Reward/loss drone trajectory shifts
Plots show **base** (dotted) vs **assembled** (solid) bird/bag sequences,
with change-point locations as vertical dashed lines.

In [3]:
main_drone = parse_js_2d_array(stimuli_src, "main_drone_position")
main_bag  = parse_js_2d_array(stimuli_src, "main_bag_position")
assert len(main_drone)==4 and len(main_bag)==4
for i in range(4):
    assert len(main_drone[i])==50 and len(main_bag[i])==50
    assert all(np.isfinite(main_drone[i])) and all(np.isfinite(main_bag[i]))
print("\u2713 Trajectory arrays: 4 blocks x 50 trials, all finite.")

HIGHVOL_SHIFT = float(re.search(r"var\s+HIGHVOL_SHIFT_DELTA\s*=\s*([\-\d.]+)", stimuli_src).group(1))
LOWVOL_SHIFT  = float(re.search(r"var\s+LOWVOL_SHIFT_DELTA\s*=\s*([\-\d.]+)", stimuli_src).group(1))

hv_base_drone = np.array(main_drone[0]); lv_base_drone = np.array(main_drone[1])
hv_base_bag  = np.array(main_bag[0]);  lv_base_bag  = np.array(main_bag[1])

def shift_seq(seq, delta): return np.clip(seq + delta, 10, 90)

hv_shifted_bird = shift_seq(hv_base_drone, HIGHVOL_SHIFT)
hv_shifted_bag  = shift_seq(hv_base_bag,  HIGHVOL_SHIFT)
lv_shifted_bird = shift_seq(lv_base_drone, LOWVOL_SHIFT)
lv_shifted_bag  = shift_seq(lv_base_bag,  LOWVOL_SHIFT)

CP_HIGH_VOL = [3, 5, 11, 19, 25, 37, 39, 47]
CP_LOW_VOL  = [3, 5, 11, 15, 19, 21, 25, 29, 31, 37, 39, 45, 47]

def plot_trajectories(reward_shifted, save_stem):
    fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True, sharey=True)
    fig.suptitle(
        f"Latent trajectories  \u00b7  reward_uses_shifted_sequences = {reward_shifted}",
        fontsize=15, fontweight="bold", y=0.98)

    # Build assembled sequences for this branch
    if reward_shifted:
        assembled_bird = [hv_shifted_bird, lv_shifted_bird, hv_base_drone, lv_base_drone]
        assembled_bag  = [hv_shifted_bag,  lv_shifted_bag,  hv_base_bag,  lv_base_bag]
        deltas = [HIGHVOL_SHIFT, LOWVOL_SHIFT, 0, 0]
    else:
        assembled_bird = [hv_base_drone, lv_base_drone, hv_shifted_bird, lv_shifted_bird]
        assembled_bag  = [hv_base_bag,  lv_base_bag,  hv_shifted_bag,  lv_shifted_bag]
        deltas = [0, 0, HIGHVOL_SHIFT, LOWVOL_SHIFT]

    bases_bird = [hv_base_drone, lv_base_drone, hv_base_drone, lv_base_drone]
    bases_bag  = [hv_base_bag,  lv_base_bag,  hv_base_bag,  lv_base_bag]

    trials = np.arange(1, 51)

    for idx in range(4):
        ax = axes[idx // 2][idx % 2]
        vol = int(factors_vol[idx])
        cps = CP_HIGH_VOL if vol == 49 else CP_LOW_VOL
        vlabel = factors_valence[idx]
        vol_str = "high" if vol == 49 else "low"
        stc_str = "high" if int(factors_stc[idx]) == 64 else "low"
        d = deltas[idx]
        shift_label = f"shifted (\u0394=+{int(d)})" if d > 0 else (f"shifted (\u0394={int(d)})" if d < 0 else "base (\u0394=+0)")

        # change-point lines
        for cp in cps:
            ax.axvline(cp, color="#999", ls="--", lw=0.7, alpha=0.55)

        # base (dotted)
        ax.plot(trials, bases_bird[idx], ls=":", lw=1.3, color="#1f77b4", alpha=0.55, label="base bird")
        ax.plot(trials, bases_bag[idx],  ls=":", lw=1.3, color="#ff7f0e", alpha=0.55, label="base bag")
        # assembled (solid)
        ax.plot(trials, assembled_bird[idx], ls="-", lw=1.8, color="#1f77b4", label="assembled bird")
        ax.plot(trials, assembled_bag[idx],  ls="-", lw=1.8, color="#ff7f0e", label="assembled bag")

        ax.set_title(
            f"design idx {idx}  \u00b7  vol={vol_str}, stc={stc_str}, valence={vlabel}\n{shift_label}",
            fontsize=11, color="#c00" if vlabel == "reward" else "#333")
        ax.set_ylim(0, 100)
        ax.set_ylabel("position (%)")
        if idx >= 2: ax.set_xlabel("trial within block")
        if idx == 0: ax.legend(fontsize=8, loc="upper right", framealpha=0.85)

    fig.tight_layout(rect=[0, 0, 1, 0.94])
    save_fig(fig, save_stem, category="game", also_html=True)
    plt.show()

plot_trajectories(True,  "trajectories_reward_shifted_true")
plot_trajectories(False, "trajectories_reward_shifted_false")
print("\u2713 Trajectory plots exported.")

✓ Trajectory arrays: 4 blocks x 50 trials, all finite.
  ✓ game_trials/trajectories_reward_shifted_true.png + .pdf
  ✓ game_trials/trajectories_reward_shifted_true.html
  ✓ game_trials/trajectories_reward_shifted_false.png + .pdf
  ✓ game_trials/trajectories_reward_shifted_false.html
✓ Trajectory plots exported.


## Change-point structure: volatility vs stochasticity

**Detection rule.** A state change = any trial where `drone[t] ≠ drone[t−1]`.

**Design intent.**
- *High vol / Low stc*: the drone makes **many large jumps** and the bag lands close to it — change points are frequent and obvious.
- *Low vol / High stc*: the drone makes **many small drifts** and the bag scatters widely — apparent variation is mostly observation noise, not real state changes.

The plots below show this contrast directly: the top row overlays drone (solid) and bag (dots) trajectories so you can see the signal-to-noise ratio at a glance; the bottom panel compares the distribution of drone jump sizes between conditions.

In [4]:
main_drone = parse_js_2d_array(stimuli_src, "main_drone_position")
main_bag   = parse_js_2d_array(stimuli_src, "main_bag_position")

# ── Panel 1: Trajectory overlays per condition ──
fig = plt.figure(figsize=(14, 8))
gs = fig.add_gridspec(2, 2, height_ratios=[2.2, 1], hspace=0.35, wspace=0.25)

cond_blocks = {
    'High vol / Low stc': [i for i in range(4) if factors_vol[i] == 49],
    'Low vol / High stc': [i for i in range(4) if factors_vol[i] == 4],
}
cond_colors = {'High vol / Low stc': '#2563eb', 'Low vol / High stc': '#dc2626'}

for ci, (cond_label, blk_indices) in enumerate(cond_blocks.items()):
    ax = fig.add_subplot(gs[0, ci])
    blk = blk_indices[0]  # show one representative block
    drone = np.array(main_drone[blk])
    bag   = np.array(main_bag[blk])
    trials = np.arange(1, 51)

    # Drone trajectory (thick step line)
    ax.step(trials, drone, where='mid', lw=2.2, color=cond_colors[cond_label],
            label='Drone (hidden state)', zorder=4)

    # Bag positions (scattered dots)
    ax.scatter(trials, bag, s=18, color='#94a3b8', alpha=0.6,
               edgecolor='white', linewidth=0.3, label='Bag (observed)', zorder=3)

    # Mark state changes with vertical ticks
    jumps = np.abs(np.diff(drone))
    cp_trials = [t+2 for t, j in enumerate(jumps) if j > 0]
    for cp in cp_trials:
        ax.axvline(cp, color=cond_colors[cond_label], alpha=0.12, lw=1, zorder=1)

    n_cp = len(cp_trials)
    mean_jump = jumps[jumps > 0].mean() if (jumps > 0).any() else 0
    bag_spread = np.abs(bag - drone).mean()

    ax.set_title(f'{cond_label}\n{n_cp} state changes · mean |Δ| = {mean_jump:.1f} · '
                 f'bag spread = {bag_spread:.1f}', fontsize=11)
    ax.set_ylabel('Position (0–100)')
    ax.set_xlabel('Trial')
    ax.set_xlim(0, 51)
    if ci == 0:
        ax.legend(fontsize=9, loc='lower left')

# ── Panel 2: Jump-size distributions ──
ax_hist = fig.add_subplot(gs[1, :])

for cond_label, blk_indices in cond_blocks.items():
    all_jumps = []
    for blk in blk_indices:
        d = np.array(main_drone[blk])
        j = np.abs(np.diff(d))
        all_jumps.extend(j[j > 0].tolist())

    bins = np.arange(0, max(all_jumps) + 2, 1.5)
    ax_hist.hist(all_jumps, bins=bins, alpha=0.55, color=cond_colors[cond_label],
                 edgecolor='white', linewidth=0.5, label=cond_label, density=False)

ax_hist.set_xlabel('|Δ drone position| per state change')
ax_hist.set_ylabel('Count')
ax_hist.set_title('Distribution of drone jump sizes by condition', fontsize=11, fontweight='bold')
ax_hist.legend(fontsize=10)

# Annotate the histograms
for cond_label, blk_indices in cond_blocks.items():
    all_jumps = []
    for blk in blk_indices:
        d = np.array(main_drone[blk])
        j = np.abs(np.diff(d))
        all_jumps.extend(j[j > 0].tolist())
    mn = np.mean(all_jumps)
    ax_hist.axvline(mn, color=cond_colors[cond_label], ls='--', lw=1.5, alpha=0.8)
    ax_hist.text(mn + 0.5, ax_hist.get_ylim()[1] * 0.85,
                 f'mean = {mn:.1f}', color=cond_colors[cond_label],
                 fontsize=9, fontweight='bold')

plt.suptitle('Drone trajectory structure: signal vs noise', fontweight='bold', fontsize=13, y=1.01)
plt.tight_layout()
save_fig(fig, "change_point_map", category="game", also_html=True)
plt.show()

# Summary table
print("\n── Summary ──")
for cond_label, blk_indices in cond_blocks.items():
    all_j = []
    for blk in blk_indices:
        d = np.array(main_drone[blk])
        j = np.abs(np.diff(d))
        all_j.extend(j[j > 0].tolist())
    bag_spreads = []
    for blk in blk_indices:
        bag_spreads.extend(np.abs(np.array(main_bag[blk]) - np.array(main_drone[blk])).tolist())
    print(f"  {cond_label}: {len(all_j)} state changes, "
          f"mean jump={np.mean(all_j):.1f}, mean bag spread={np.mean(bag_spreads):.1f}")


  ✓ game_trials/change_point_map.png + .pdf
  ✓ game_trials/change_point_map.html

── Summary ──
  High vol / Low stc: 30 state changes, mean jump=4.0, mean bag spread=4.8
  Low vol / High stc: 30 state changes, mean jump=14.2, mean bag spread=4.6


## K. Drop-object distribution and feedback audit

In [5]:
src_drop = details_src if details_src else stimuli_src
drop_dist = parse_js_flat_array(src_drop, "drop_obj_distribution_default")
drop_dur  = parse_js_flat_array(src_drop, "drop_obj_duration_default")

drop_df = pd.DataFrame({"fragment": range(len(drop_dist)),
    "h_offset": drop_dist, "duration_ms": [int(d) for d in drop_dur]})
display(drop_df)

assert len(drop_dist)==10 and len(drop_dur)==10
assert all(np.isfinite(drop_dist)) and all(np.isfinite(drop_dur))
print("\u2713 10 fragments, matching lengths, all finite.")

fb_df = pd.DataFrame({"captured": range(11),
    "reward_fb": range(11), "loss_fb": [c-10 for c in range(11)]})
display(fb_df)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.5))
ax1.bar(drop_df.fragment, drop_df.h_offset, color="#1f77b4", alpha=0.8)
ax1.set_xlabel("Fragment index"); ax1.set_ylabel("Horizontal offset (pp)")
ax1.set_title("Drop-object horizontal distribution", fontweight="bold")
ax2.bar(drop_df.fragment, drop_df.duration_ms, color="#ff7f0e", alpha=0.8)
ax2.set_xlabel("Fragment index"); ax2.set_ylabel("Duration (ms)")
ax2.set_title("Drop-object fall durations", fontweight="bold")
fig.tight_layout()
save_fig(fig, "drop_object_distribution", category="game", also_html=True)
plt.show()

save_html(
    styled_table_css("dropfb") + '<div id="dropfb">' +
    "<table><caption>Drop-object distribution</caption>" + drop_df.to_html(index=False) + "</table>" +
    "<table><caption>Reward / loss feedback mapping</caption>" + fb_df.to_html(index=False) + "</table></div>", "drop_object_distribution_feedback_audit.html", category="game")

,fragment,h_offset,duration_ms
0,0,-4.25,350
1,1,-3.00,400
2,2,-1.75,500
3,3,-0.75,550
4,4,-0.25,600
5,5,0.25,600
6,6,0.75,600
7,7,1.75,500
8,8,3.00,500
9,9,4.25,400


✓ 10 fragments, matching lengths, all finite.


,captured,reward_fb,loss_fb
0,0,0,-10
1,1,1,-9
2,2,2,-8
3,3,3,-7
4,4,4,-6
5,5,5,-5
6,6,6,-4
7,7,7,-3
8,8,8,-2
9,9,9,-1


  ✓ game_trials/drop_object_distribution.png + .pdf
  ✓ game_trials/drop_object_distribution.html
  ✓ game_trials/drop_object_distribution_feedback_audit.html


## Reward scoring function

Step-function thresholds parsed directly from `trial.js`. Left panel: catch count vs distance. Right panel: 50k Monte Carlo L/R symmetry check.

In [6]:
thresholds = parse.capture_thresholds(trial_src)
def compute_capture(d):
    for t, c in thresholds[:-1]:
        if d <= t: return c
    return 0

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
d = np.linspace(0, 20, 2001)
scores = [compute_capture(x) for x in d]
axes[0].plot(d, scores, lw=2.2, color='#2563eb')
axes[0].fill_between(d, scores, alpha=.1, color='#2563eb')
for t, c in thresholds[:-1]:
    axes[0].plot(t, c, 'o', ms=5, color='#2563eb', zorder=5)
axes[0].set_xlabel('|bag − bucket|'); axes[0].set_ylabel('Fragments caught')
axes[0].set_title('Catch function'); axes[0].set_yticks(range(0, 11))

np.random.seed(42)
bag = np.random.uniform(8, 92, 50000)
bkt = np.random.uniform(8, 92, 50000)
signed = bag - bkt
caught = np.array([compute_capture(abs(x)) for x in signed])
bins = np.arange(-25, 26, 1)
idx = np.digitize(signed, bins)
means = [caught[idx==i].mean() if (idx==i).sum()>0 else np.nan for i in range(1, len(bins))]
centres = (bins[:-1] + bins[1:]) / 2
axes[1].bar(centres, means, width=.9, color='#2563eb', alpha=.7)
axes[1].axvline(0, color='#dc2626', lw=1, ls='--')
axes[1].set_xlabel('Signed distance (bag − bucket)')
axes[1].set_ylabel('Mean fragments caught')
axes[1].set_title('L/R symmetry (50k draws)')
plt.tight_layout()
save_fig(fig, 'scoring_function', 'game')
plt.show()

left_m = caught[signed < -2].mean(); right_m = caught[signed > 2].mean()
print(f'Left: {left_m:.4f}, Right: {right_m:.4f}, Δ={abs(left_m-right_m):.4f}')
print('✅ No asymmetry.' if abs(left_m-right_m) < .05 else '⚠ Investigate.')

  ✓ game_trials/scoring_function.png + .pdf
Left: 1.4712, Right: 1.4837, Δ=0.0125
✅ No asymmetry.
